# Hypothesis 4: Poverty as a Push Factor for Emigration Volume

This notebook tests whether Sri Lanka's poverty rate is linearly associated with emigration volume using Pearson correlation.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.stats import pearsonr, norm, t

In [ ]:
CSV_PATH = Path("SriLanka_Migration_Dinura_Chanupa.csv")
YEAR_COL = "year"
POVERTY_COL = "poverty_rate_annual"
EMIGRATION_COL = "slbfe_total_annual"

In [ ]:
df = pd.read_csv(CSV_PATH)
required_cols = [YEAR_COL, POVERTY_COL, EMIGRATION_COL]
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

data = df[required_cols].dropna().copy()
data[YEAR_COL] = data[YEAR_COL].astype(int)
data[POVERTY_COL] = data[POVERTY_COL].astype(float)
data[EMIGRATION_COL] = data[EMIGRATION_COL].astype(float)

if len(data) < 4:
    raise ValueError("Need at least 4 observations for this analysis.")

print(f"Rows used: {len(data)}")
data.head()

In [ ]:
x = data[POVERTY_COL].to_numpy(dtype=float)
y = data[EMIGRATION_COL].to_numpy(dtype=float)
n = len(data)

r, p_two_tailed = pearsonr(x, y)
dfree = n - 2
t_stat = float(r * np.sqrt(dfree / (1 - r**2)))
p_one_tailed_negative = float(t.cdf(t_stat, df=dfree))
r_squared = float(r**2)

z = np.arctanh(float(r))
se = 1 / np.sqrt(n - 3)
z_critical = norm.ppf(0.975)
ci_low = float(np.tanh(z - z_critical * se))
ci_high = float(np.tanh(z + z_critical * se))

print("Pearson correlation results")
print(f"n = {n}")
print(f"r = {r:.4f}")
print(f"t = {t_stat:.4f}, df = {dfree}")
print(f"p (two-tailed) = {p_two_tailed:.6f}")
print(f"p (one-tailed, rho < 0) = {p_one_tailed_negative:.6f}")
print(f"95% CI for r = [{ci_low:.4f}, {ci_high:.4f}]")
print(f"R^2 = {r_squared:.4f}")

In [ ]:
fig, ax1 = plt.subplots(figsize=(11, 6))
ax2 = ax1.twinx()

ax1.plot(data[YEAR_COL], data[POVERTY_COL], color="#2E4057", marker="o", linewidth=2)
ax2.plot(data[YEAR_COL], data[EMIGRATION_COL], color="#C73E1D", marker="s", linewidth=2)

ax1.set_title("Figure 01: Poverty Rate vs Emigration Volume Time Series (1994-2025)")
ax1.set_xlabel("Year")
ax1.set_ylabel("Poverty Rate (%)", color="#2E4057")
ax2.set_ylabel("Emigration Volume (SLBFE Total Annual)", color="#C73E1D")
ax1.grid(alpha=0.25)
fig.tight_layout()
plt.show()

In [ ]:
slope, intercept = np.polyfit(x, y, 1)
x_line = np.linspace(x.min(), x.max(), 200)
y_line = slope * x_line + intercept

fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(x, y, color="#1B998B", edgecolor="white", linewidth=0.6, s=55, alpha=0.9)
ax.plot(x_line, y_line, color="#111111", linewidth=2)
ax.set_title("Figure 02: Poverty Rate vs Emigration Volume (Scatter + Regression)")
ax.set_xlabel("Poverty Rate (%)")
ax.set_ylabel("Emigration Volume (SLBFE Total Annual)")
ax.grid(alpha=0.25)
ax.text(
    0.03,
    0.96,
    f"r = {r:.3f}\np(two-tailed) = {p_two_tailed:.4f}",
    transform=ax.transAxes,
    verticalalignment="top",
    bbox={"boxstyle": "round", "facecolor": "white", "alpha": 0.75, "edgecolor": "#AAAAAA"},
)
fig.tight_layout()
plt.show()

In [ ]:
if p_two_tailed < 0.05:
    decision = "Reject H0"
    interpretation = (
        "There is a statistically significant linear relationship between poverty rate and emigration volume."
    )
else:
    decision = "Fail to reject H0"
    interpretation = (
        "The primary two-tailed test does not provide sufficient evidence of a significant linear relationship."
    )

print("Hypothesis test decision")
print(f"Decision: {decision}")
print(f"Interpretation: {interpretation}")